# GPS-Enhanced Horse Race Simulator

A collaborative portfolio project comparing traditional past-performance features with GPS-derived race features, then visualizing race scenarios in an interactive Dash app.

Developed by Ethan Brown and Aiden.

## Data required (not included)

Place authorized copies of these CSV files in the repository `data/` directory before running the notebook:

- `GPS Races (December 24, 2025 – March 24, 2026).csv`
- `Starters PPs (relate to GPS Races).csv`
- `startersgpspps.csv`
- `upcoming_races.csv`
- `upcomingstarterspps.csv`

The datasets, project brief, and saved outputs are intentionally excluded. See `data/README.md` and the repository README for data-handling guidance.

This is a portfolio demonstration, not betting advice. Model results are not a guarantee of future performance.

## 0 · Environment

In [ ]:
# Install once from the repository root: python -m pip install -r requirements.txt

## 1 · Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

import dash
from dash import dcc, html, Input, Output, State
import plotly.express as px
import plotly.graph_objects as go

print(f"pandas {pd.__version__} | numpy {np.__version__}")

## 2 · Helper Functions

All GPS feature columns after `build_pp_gps_features()` are named exactly as they
come out of `summarize_gps_race_path()` — e.g. `gps_avg_time_behind`, `gps_stride_eff`.
There is **no extra prefix added** during aggregation.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 · HELPERS
# ─────────────────────────────────────────────────────────────────────────────

RACE_KEYS = ["track_id", "race_dt", "race_number"]

HORSE_COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990",
    "#dcbeff", "#9a6324",
]

# ── Scalar utilities ──────────────────────────────────────────────────────────

def parse_money(series: pd.Series) -> pd.Series:
    """Strip $ / commas → float."""
    return (
        series.astype(str)
        .str.replace(r"[$, ]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .astype(float)
    )


def frac_to_decimal(x) -> float:
    """'9/2' → 5.5  |  '2.5' → 2.5  |  NaN → NaN"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in {"", "nan", "None"}:
        return np.nan
    if "/" in s:
        try:
            a, b = s.split("/")
            return 1.0 + float(a) / float(b)
        except Exception:
            return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def safe_z(series: pd.Series) -> pd.Series:
    """Z-score; returns zeros when std == 0 or all-NaN."""
    s = pd.to_numeric(series, errors="coerce").astype(float)
    std = s.std()
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std


def softmax(x) -> np.ndarray:
    """Numerically stable softmax."""
    arr = np.array(x, dtype=float)
    arr -= np.nanmax(arr)
    ex = np.exp(arr)
    total = np.nansum(ex)
    if total == 0 or np.isnan(total):
        return np.full(len(arr), 1.0 / len(arr))
    return ex / total


def prep_dates(df: pd.DataFrame, date_col: str = "race_date") -> pd.DataFrame:
    """Strip column whitespace + parse race_date → race_dt."""
    out = df.copy()
    out.columns = out.columns.str.strip()
    out["race_dt"] = pd.to_datetime(out[date_col], format="%m/%d/%y", errors="coerce")
    return out


# ── GPS path summarisation ────────────────────────────────────────────────────

def summarize_gps_race_path(grp: pd.DataFrame) -> pd.Series:
    """
    Collapse per-gate GPS rows for one horse-race → single summary row.

    Gate numbering: highest gate = race start, gate 0 = finish.
    Race divided into early / mid / late thirds by gate order (descending).

    Output column names (no extra prefix — just what you see here):
        gps_finish, gps_finish_time, gps_total_distance_ran, gps_total_strides,
        gps_avg_time_behind, gps_avg_dist_behind,
        gps_early_pos, gps_mid_pos, gps_late_pos,
        gps_early_time_behind, gps_mid_time_behind, gps_late_time_behind,
        gps_avg_sectional, gps_stride_eff, gps_close_gain, gps_time_gain
    """
    grp = grp.sort_values("gate", ascending=False).copy()
    n = len(grp)
    thirds = np.array_split(np.arange(n), 3)
    early, mid, late = grp.iloc[thirds[0]], grp.iloc[thirds[1]], grp.iloc[thirds[2]]

    finish = grp.loc[grp["gate"].eq(0)].head(1)
    if finish.empty:
        finish = grp.tail(1)

    td = finish["cumulative_distance_ran"].iloc[0]
    ts = finish["cumulative_strides"].iloc[0]
    stride_eff = td / ts if (pd.notna(td) and pd.notna(ts) and ts > 0) else np.nan

    return pd.Series({
        "gps_finish":             finish["official_position"].iloc[0],
        "gps_finish_time":        finish["running_time"].iloc[0],
        "gps_total_distance_ran": td,
        "gps_total_strides":      ts,
        "gps_avg_time_behind":    grp["time_behind"].mean(),
        "gps_avg_dist_behind":    grp["distance_behind"].mean(),
        "gps_early_pos":          early["position"].mean(),
        "gps_mid_pos":            mid["position"].mean(),
        "gps_late_pos":           late["position"].mean(),
        "gps_early_time_behind":  early["time_behind"].mean(),
        "gps_mid_time_behind":    mid["time_behind"].mean(),
        "gps_late_time_behind":   late["time_behind"].mean(),
        "gps_avg_sectional":      grp["sectional_time"].mean(),
        "gps_stride_eff":         stride_eff,
        "gps_close_gain":         float(early["position"].mean()) - float(late["position"].mean()),
        "gps_time_gain":          float(early["time_behind"].mean()) - float(late["time_behind"].mean()),
    })


# ── PP feature builders ───────────────────────────────────────────────────────

def build_pp_traditional_features(pp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate up-to-3 traditional past performances → one row per horse.

    Output columns (prefix pp_):
        pp_starts, pp_avg_finish, pp_win_rate, pp_top3_rate,
        pp_avg_post, pp_avg_odds, pp_avg_mline, pp_avg_distance,
        pp_avg_poc1..5, pp_avg_len_fin,
        pp_last_track, pp_last_surface, pp_last_race_type, pp_last_pos
    """
    df = pp_df.copy()

    # Normalise the Post_time_Odds casing difference between files
    if "Post_time_Odds" in df.columns and "post_time_odds" not in df.columns:
        df = df.rename(columns={"Post_time_Odds": "post_time_odds"})

    # Purse (strip spaces from column name already done by prep_dates,
    # but the column may be 'Purse' with capital P in some files)
    purse_col = next((c for c in df.columns if c.lower() == "purse"), None)
    df["purse_num"] = parse_money(df[purse_col]) if purse_col else np.nan

    numeric_cols = [
        "post_position", "official_position", "post_time_odds", "field_size", "distance",
        "position_at_point_of_call_1", "position_at_point_of_call_2",
        "position_at_point_of_call_3", "position_at_point_of_call_4",
        "position_at_point_of_call_5",
        "length_behind_at_poc_1", "length_behind_at_poc_2",
        "length_behind_at_poc_3", "length_behind_at_poc_4",
        "length_behind_at_poc_5", "length_behind_at_finish",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = np.nan

    df["ml_dec"] = df["morning_line_odds"].map(frac_to_decimal)
    df["won"]    = (df["official_position"] == 1).astype(float)
    df["top3"]   = (df["official_position"] <= 3).astype(float)

    df = df.sort_values(["registration_number", "race_dt", "race_number"]).reset_index(drop=True)

    agg = df.groupby("registration_number").agg(
        pp_starts         = ("official_position",              "count"),
        pp_avg_finish     = ("official_position",              "mean"),
        pp_win_rate       = ("won",                            "mean"),
        pp_top3_rate      = ("top3",                           "mean"),
        pp_avg_post       = ("post_position",                  "mean"),
        pp_avg_odds       = ("post_time_odds",                 "mean"),
        pp_avg_mline      = ("ml_dec",                         "mean"),
        pp_avg_distance   = ("distance",                       "mean"),
        pp_avg_poc1       = ("position_at_point_of_call_1",    "mean"),
        pp_avg_poc2       = ("position_at_point_of_call_2",    "mean"),
        pp_avg_poc3       = ("position_at_point_of_call_3",    "mean"),
        pp_avg_poc4       = ("position_at_point_of_call_4",    "mean"),
        pp_avg_poc5       = ("position_at_point_of_call_5",    "mean"),
        pp_avg_len_fin    = ("length_behind_at_finish",        "mean"),
        pp_last_track     = ("track_id",                       "last"),
        pp_last_surface   = ("surface",                        "last"),
        pp_last_race_type = ("race_type",                      "last"),
        pp_last_pos       = ("official_position",              "last"),
    ).reset_index()

    return agg


def build_pp_gps_features(gps_pp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarise GPS gate-level PP rows → one row per horse.

    Steps:
      1. Cast numerics.
      2. Collapse per-gate rows → one row per horse-race via summarize_gps_race_path().
      3. Average all race summaries per horse.

    Output columns (no extra prefix beyond what summarize_gps_race_path() produces):
        gps_finish, gps_finish_time, gps_total_distance_ran, gps_total_strides,
        gps_avg_time_behind, gps_avg_dist_behind,
        gps_early_pos, gps_mid_pos, gps_late_pos,
        gps_early_time_behind, gps_mid_time_behind, gps_late_time_behind,
        gps_avg_sectional, gps_stride_eff, gps_close_gain, gps_time_gain,
        gps_pp_count
    """
    df = gps_pp_df.copy()

    for col in ["gate", "position", "official_position", "sectional_time", "running_time",
                "time_behind", "distance_behind", "distance_ran",
                "cumulative_distance_ran", "strides", "cumulative_strides"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = prep_dates(df)

    keys = [k for k in ["registration_number", "race_dt", "race_number", "track_id"]
            if k in df.columns]
    race_level = (
        df.groupby(keys, dropna=False)
        .apply(summarize_gps_race_path)
        .reset_index()
    )

    gps_cols = [c for c in race_level.columns if c.startswith("gps_")]
    agg = race_level.groupby("registration_number")[gps_cols].mean().reset_index()
    agg["gps_pp_count"] = race_level.groupby("registration_number").size().values

    return agg   # columns: registration_number, gps_finish, gps_finish_time, ...


# ── Model factory ─────────────────────────────────────────────────────────────

def make_classifier(
    numeric_features: list,
    categorical_features: list,
    random_state: int = 42,
) -> Pipeline:
    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]),
             numeric_features),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), categorical_features),
        ]
    )
    return Pipeline([
        ("prep", preprocess),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        )),
    ])


# ── Race-relative ranking ─────────────────────────────────────────────────────

def add_race_relative_features(
    df: pd.DataFrame,
    race_keys: list,
    spec: list,
) -> pd.DataFrame:
    """
    For each (col, ascending) pair in spec, add a {col}_race_rank percentile column
    (0 = worst in field, 1 = best).  ascending=True means lower raw value is better.
    Only adds a column if `col` actually exists in df.
    """
    out = df.copy()
    for col, ascending in spec:
        if col in out.columns:
            out[f"{col}_race_rank"] = out.groupby(race_keys)[col].rank(
                pct=True, ascending=ascending, na_option="keep"
            )
    return out


# ── Horse explanation ─────────────────────────────────────────────────────────

def make_horse_explanation(row: pd.Series) -> str:
    bits = []
    gps_adv = float(row.get("gps_advantage", 0) or 0)
    if gps_adv > 0.04:
        bits.append("GPS model is noticeably more bullish than the traditional model")
    elif gps_adv < -0.04:
        bits.append("GPS model is more cautious than the traditional model")
    if float(row.get("closing_speed", 0) or 0) > 0.2:
        bits.append("strong late-closing profile")
    if float(row.get("early_speed", 0) or 0) > 0.2:
        bits.append("projects as an early pace factor")
    if float(row.get("gps_stride_efficiency", 0) or 0) > 0:
        bits.append("above-average stride efficiency")
    if not float(row.get("gps_pp_count", 0) or 0):
        bits.append("no GPS past performances — traditional model only")
    if not bits:
        bits.append("neutral across pace and efficiency indicators")
    return "; ".join(bits).capitalize() + "."


print("✅  All helper functions defined.")

## 3 · Load Data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 · LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = Path("data")

# Race outcomes (targets for model training)
traditional_races = prep_dates(
    pd.read_csv(DATA_DIR / "GPS Races (December 24, 2025 – March 24, 2026).csv",
                low_memory=False)
)

# Past-performance history for horses that ran in the GPS window
#   'gps_date' column = 'X' means that PP row also has GPS gate data
starters_pps = prep_dates(
    pd.read_csv(DATA_DIR / "Starters PPs (relate to GPS Races).csv", low_memory=False)
)
starters_pps = starters_pps.rename(columns={"gps_date": "gps_data_flag"})

# GPS gate-level telemetry for flagged PPs (and for upcoming horses' PPs)
gps_pps = prep_dates(
    pd.read_csv(DATA_DIR / "startersgpspps.csv", low_memory=False)
)

# Upcoming race card
upcoming_races = prep_dates(
    pd.read_csv(DATA_DIR / "upcoming_races.csv", low_memory=False)
)

# PPs for upcoming horses
upcoming_pps = prep_dates(
    pd.read_csv(DATA_DIR / "upcomingstarterspps.csv", low_memory=False)
)
upcoming_pps = upcoming_pps.rename(columns={"gps_date": "gps_data_flag"})

print(f"traditional_races : {traditional_races.shape[0]:,} rows | {traditional_races['registration_number'].nunique():,} horses")
print(f"starters_pps      : {starters_pps.shape[0]:,} rows | {starters_pps['registration_number'].nunique():,} horses")
print(f"  └ GPS-flagged PPs: {(starters_pps['gps_data_flag']=='X').sum():,}")
print(f"gps_pps           : {gps_pps.shape[0]:,} rows | {gps_pps['registration_number'].nunique():,} horses")
print(f"upcoming_races    : {upcoming_races.shape[0]:,} rows | {upcoming_races['registration_number'].nunique():,} horses")
print(f"upcoming_pps      : {upcoming_pps.shape[0]:,} rows | {upcoming_pps['registration_number'].nunique():,} horses")

## 4 · Feature Engineering

### Design
We aggregate each horse's available PPs (up to 3) into a single feature snapshot.
This is leak-free: the snapshot only uses races *before* the one being predicted.

### Verified column names
After running `build_pp_gps_features()`, GPS columns are named exactly as they
appear in `summarize_gps_race_path()` output: `gps_avg_time_behind`, `gps_stride_eff`,
`gps_close_gain`, etc. — no extra prefix.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 · FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────

trad_profiles = build_pp_traditional_features(starters_pps)
gps_profiles  = build_pp_gps_features(gps_pps)

print(f"Traditional PP profiles : {len(trad_profiles):,} horses")
print(f"GPS PP profiles         : {len(gps_profiles):,} horses")
print()

# ── Build model frame ──────────────────────────────────────────────────────────
model_df = traditional_races.copy()

# Safety: strip column names in case prep_dates was not called or file differs
model_df.columns = model_df.columns.str.strip()

# Diagnostic: confirm expected columns are present
_expected = ["post_time_odds", "morning_line_odds", "official_position",
             "registration_number", "track_id", "race_number", "race_dt"]
_missing  = [c for c in _expected if c not in model_df.columns]
if _missing:
    print(f"WARNING: missing expected columns: {_missing}")
    print(f"Available columns: {list(model_df.columns)}")
    # If race_dt missing, re-apply prep_dates
    if "race_dt" not in model_df.columns and "race_date" in model_df.columns:
        model_df["race_dt"] = pd.to_datetime(model_df["race_date"], format="%m/%d/%y", errors="coerce")
        print("  -> re-parsed race_dt from race_date")
else:
    print("All expected columns present.")

purse_col = next((c for c in model_df.columns if c.lower() == "purse"), None)
model_df["purse_num"] = parse_money(model_df[purse_col]) if purse_col else np.nan

model_df["ml_dec"] = model_df["morning_line_odds"].map(frac_to_decimal)
model_df["won"]    = (model_df["official_position"] == 1).astype(int)
model_df["top3"]   = (model_df["official_position"] <= 3).astype(int)

# post_time_odds may be named differently in some versions of the file
if "post_time_odds" in model_df.columns:
    model_df["post_time_odds"] = pd.to_numeric(model_df["post_time_odds"], errors="coerce")
elif "Post_time_Odds" in model_df.columns:
    model_df["post_time_odds"] = pd.to_numeric(model_df["Post_time_Odds"], errors="coerce")
else:
    model_df["post_time_odds"] = np.nan
    print("  NOTE: post_time_odds not found, filling with NaN")

model_df["field_size_calc"] = model_df.groupby(RACE_KEYS)["registration_number"].transform("size")

model_df = (
    model_df
    .merge(trad_profiles, on="registration_number", how="left")
    .merge(gps_profiles,  on="registration_number", how="left")
)

print(f"model_df shape : {model_df.shape}")
print(f"GPS coverage   : {model_df['gps_pp_count'].notna().mean():.1%} of starts have GPS PP history")
print()

RANK_SPEC = [
    ("ml_dec",              True),
    ("pp_avg_finish",       True),
    ("pp_win_rate",         False),
    ("pp_top3_rate",        False),
    ("gps_avg_time_behind", True),
    ("gps_close_gain",      False),
    ("gps_stride_eff",      False),
]
model_df = add_race_relative_features(model_df, RACE_KEYS, RANK_SPEC)

model_df[[
    "horse_name", "track_id", "race_dt", "race_number",
    "pp_starts", "pp_avg_finish", "pp_win_rate",
    "gps_pp_count", "gps_avg_time_behind",
]].head(8)


## 5 · Model Training & Evaluation

Four Random Forest pipelines (median imputation → OHE → RF):

| Model | Features | Target |
|-------|----------|--------|
| Traditional Win   | `pp_*` + race ranks | `won` |
| GPS Win           | `pp_*` + `gps_*` + race ranks | `won` |
| Traditional Top-3 | `pp_*` + race ranks | `top3` |
| GPS Top-3         | `pp_*` + `gps_*` + race ranks | `top3` |

Chronological 80/20 split — trains on earlier races, tests on the most recent 20%.
GPS features will be NaN for horses without GPS PPs; the median imputer handles this.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 · MODEL TRAINING & EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

# Safety strip in case columns have whitespace
model_df.columns = model_df.columns.str.strip()

# Traditional (non-GPS) features
COMMON_NUMERIC = [
    "post_position", "ml_dec", "field_size_calc",
    "pp_starts", "pp_avg_finish", "pp_win_rate", "pp_top3_rate",
    "pp_avg_post", "pp_avg_odds", "pp_avg_mline",
    "pp_avg_distance", "pp_avg_poc1", "pp_avg_poc2", "pp_avg_poc3",
    "pp_avg_poc4", "pp_avg_poc5", "pp_avg_len_fin",
    # race-relative ranks (traditional)
    "ml_dec_race_rank", "pp_avg_finish_race_rank",
    "pp_win_rate_race_rank", "pp_top3_rate_race_rank",
]

# GPS-only features (verified column names from build_pp_gps_features output)
GPS_ONLY_NUMERIC = [
    "gps_pp_count",
    "gps_finish", "gps_finish_time",
    "gps_total_distance_ran", "gps_total_strides",
    "gps_avg_time_behind", "gps_avg_dist_behind",
    "gps_early_pos", "gps_mid_pos", "gps_late_pos",
    "gps_early_time_behind", "gps_mid_time_behind", "gps_late_time_behind",
    "gps_avg_sectional", "gps_stride_eff",
    "gps_close_gain", "gps_time_gain",
    # race-relative ranks (GPS)
    "gps_avg_time_behind_race_rank",
    "gps_close_gain_race_rank",
    "gps_stride_eff_race_rank",
]

GPS_NUMERIC = COMMON_NUMERIC + GPS_ONLY_NUMERIC

CATEGORICAL_FEATURES = [
    "track_id", "surface",
    "pp_last_track", "pp_last_surface", "pp_last_race_type",
]

# Filter to columns that actually exist in model_df
COMMON_NUMERIC       = [c for c in COMMON_NUMERIC       if c in model_df.columns]
GPS_NUMERIC          = [c for c in GPS_NUMERIC           if c in model_df.columns]
GPS_ONLY_NUMERIC     = [c for c in GPS_ONLY_NUMERIC      if c in model_df.columns]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in model_df.columns]

# Cast types
for col in set(COMMON_NUMERIC + GPS_NUMERIC):
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")
for col in CATEGORICAL_FEATURES:
    model_df[col] = model_df[col].astype(str).str.strip().replace({"nan": np.nan})

# Chronological split
cutoff = model_df["race_dt"].quantile(0.80)
train_df = model_df[model_df["race_dt"] <  cutoff].copy()
test_df  = model_df[model_df["race_dt"] >= cutoff].copy()
print(f"Train : {len(train_df):,} rows  ({train_df['race_dt'].min().date()} → {train_df['race_dt'].max().date()})")
print(f"Test  : {len(test_df):,} rows   ({test_df['race_dt'].min().date()} → {test_df['race_dt'].max().date()})")
print()

trad_win_model  = make_classifier(COMMON_NUMERIC, CATEGORICAL_FEATURES)
gps_win_model   = make_classifier(GPS_NUMERIC,    CATEGORICAL_FEATURES)
trad_top3_model = make_classifier(COMMON_NUMERIC, CATEGORICAL_FEATURES)
gps_top3_model  = make_classifier(GPS_NUMERIC,    CATEGORICAL_FEATURES)

for name, model, feats, target in [
    ("Traditional Win",  trad_win_model,  COMMON_NUMERIC + CATEGORICAL_FEATURES, "won"),
    ("GPS Win",          gps_win_model,   GPS_NUMERIC    + CATEGORICAL_FEATURES, "won"),
    ("Traditional Top3", trad_top3_model, COMMON_NUMERIC + CATEGORICAL_FEATURES, "top3"),
    ("GPS Top3",         gps_top3_model,  GPS_NUMERIC    + CATEGORICAL_FEATURES, "top3"),
]:
    model.fit(train_df[feats], train_df[target])
    print(f"  ✓ Trained: {name}")

# Evaluate
records = []
for name, model, feats, target in [
    ("Traditional Win",  trad_win_model,  COMMON_NUMERIC + CATEGORICAL_FEATURES, "won"),
    ("GPS Win",          gps_win_model,   GPS_NUMERIC    + CATEGORICAL_FEATURES, "won"),
    ("Traditional Top3", trad_top3_model, COMMON_NUMERIC + CATEGORICAL_FEATURES, "top3"),
    ("GPS Top3",         gps_top3_model,  GPS_NUMERIC    + CATEGORICAL_FEATURES, "top3"),
]:
    probs = model.predict_proba(test_df[feats])[:, 1]
    preds = (probs >= 0.5).astype(int)
    records.append({
        "Model":    name,
        "AUC-ROC":  round(roc_auc_score(test_df[target], probs), 4),
        "Accuracy": round(accuracy_score(test_df[target], preds), 4),
        "Log-Loss": round(log_loss(test_df[target], probs), 4),
    })

metrics_df = pd.DataFrame(records).set_index("Model")
print("\n── Test-set metrics ──────────────────────────────────────────────────────")
display(metrics_df)

## 6 · Score Upcoming Races

GPS PPs for upcoming horses live in the same `startersgpspps.csv` file —
we just filter it to registration numbers that appear in `upcoming_races.csv`.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 · SCORE UPCOMING RACES
# ─────────────────────────────────────────────────────────────────────────────

up_trad_profiles = build_pp_traditional_features(upcoming_pps)

upcoming_reg     = set(upcoming_races["registration_number"].unique())
up_gps_subset    = gps_pps[gps_pps["registration_number"].isin(upcoming_reg)]
up_gps_profiles  = build_pp_gps_features(up_gps_subset)

print(f"Upcoming trad profiles : {len(up_trad_profiles):,} horses")
print(f"Upcoming GPS profiles  : {len(up_gps_profiles):,} horses ({len(up_gps_profiles)/len(upcoming_reg):.1%} coverage)")
print()

upcoming_df = upcoming_races.copy()
upcoming_df["ml_dec"] = upcoming_df["morning_line_odds"].map(frac_to_decimal)
upcoming_df["field_size_calc"] = upcoming_df.groupby(RACE_KEYS)["registration_number"].transform("size")

# Pull surface from PPs (upcoming_races doesn't include it)
if "surface" not in upcoming_df.columns:
    surface_map = upcoming_pps.groupby("registration_number")["surface"].last()
    upcoming_df["surface"] = upcoming_df["registration_number"].map(surface_map)

upcoming_df = (
    upcoming_df
    .merge(up_trad_profiles, on="registration_number", how="left")
    .merge(up_gps_profiles,  on="registration_number", how="left")
)
upcoming_df = add_race_relative_features(upcoming_df, RACE_KEYS, RANK_SPEC)

for col in set(COMMON_NUMERIC + GPS_NUMERIC):
    if col in upcoming_df.columns:
        upcoming_df[col] = pd.to_numeric(upcoming_df[col], errors="coerce")
for col in CATEGORICAL_FEATURES:
    if col in upcoming_df.columns:
        upcoming_df[col] = upcoming_df[col].astype(str).str.strip().replace({"nan": np.nan})

# Score with all four models
upcoming_df["trad_win_raw"]  = trad_win_model.predict_proba( upcoming_df[COMMON_NUMERIC + CATEGORICAL_FEATURES])[:, 1]
upcoming_df["gps_win_raw"]   = gps_win_model.predict_proba(  upcoming_df[GPS_NUMERIC    + CATEGORICAL_FEATURES])[:, 1]
upcoming_df["trad_top3_raw"] = trad_top3_model.predict_proba(upcoming_df[COMMON_NUMERIC + CATEGORICAL_FEATURES])[:, 1]
upcoming_df["gps_top3_raw"]  = gps_top3_model.predict_proba( upcoming_df[GPS_NUMERIC    + CATEGORICAL_FEATURES])[:, 1]

# Normalise within each race
for raw_col in ["trad_win_raw", "gps_win_raw", "trad_top3_raw", "gps_top3_raw"]:
    prob_col = raw_col.replace("_raw", "_prob")
    upcoming_df[prob_col] = upcoming_df.groupby(RACE_KEYS)[raw_col].transform(
        lambda s: s / s.sum() if s.sum() > 0 else np.full(len(s), 1.0 / len(s))
    )

upcoming_df["gps_advantage"]    = upcoming_df["gps_win_prob"] - upcoming_df["trad_win_prob"]
upcoming_df["fair_odds_decimal"] = 1.0 / upcoming_df["gps_win_prob"].clip(lower=1e-6)
upcoming_df["fair_odds_net"]     = upcoming_df["fair_odds_decimal"] - 1.0

print(f"Scored {len(upcoming_df):,} starts across {upcoming_df.groupby(RACE_KEYS).ngroups} races")
upcoming_df[[
    "track_id", "race_dt", "race_number", "horse_name",
    "trad_win_prob", "gps_win_prob", "gps_advantage",
]].sort_values(["race_dt", "race_number", "gps_win_prob"], ascending=[True, True, False]).head(12)

## 7 · Simulator Feature Derivation

| Feature | Derivation | Meaning |
|---------|-----------|---------|
| `top_speed` | `1 / gps_finish_time` | Inverse finish time → speed proxy |
| `early_speed` | `1 / (1 + pp_avg_poc1)` | Early positional advantage |
| `closing_speed` | `pp_avg_poc1 − pp_avg_finish` | Places gained from 1st call to wire |
| `track_fit` | `1/(1+pp_avg_finish) + 0.5·pp_win_rate` | Quality + win rate blend |
| `gps_burst` | `1 / (1 + gps_avg_time_behind)` | GPS gap to leader |
| `gps_stride_efficiency` | `gps_stride_eff` | Metres per stride |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 · SIMULATOR FEATURE DERIVATION
# ─────────────────────────────────────────────────────────────────────────────

dash_df = upcoming_df.copy()

dash_df["race_label"] = (
    dash_df["race_dt"].dt.strftime("%Y-%m-%d") + "  |  "
    + dash_df["track_id"].astype(str).str.strip()
    + "  |  Race " + dash_df["race_number"].astype(str)
)

# Interpretable pace / efficiency features
# Column names are exactly as produced by build_pp_gps_features()
dash_df["top_speed"] = (
    1.0 / dash_df["gps_finish_time"].clip(lower=1e-6)
    if "gps_finish_time" in dash_df.columns else np.nan
)
dash_df["gps_burst"] = (
    1.0 / (1.0 + dash_df["gps_avg_time_behind"].clip(lower=0))
    if "gps_avg_time_behind" in dash_df.columns else np.nan
)
dash_df["gps_stride_efficiency"] = (
    dash_df["gps_stride_eff"]
    if "gps_stride_eff" in dash_df.columns else np.nan
)

dash_df["early_speed"] = (
    1.0 / (1.0 + dash_df["pp_avg_poc1"].clip(lower=0))
    if "pp_avg_poc1" in dash_df.columns else np.nan
)
dash_df["closing_speed"] = (
    dash_df["pp_avg_poc1"] - dash_df["pp_avg_finish"]
    if ("pp_avg_poc1" in dash_df.columns and "pp_avg_finish" in dash_df.columns) else np.nan
)
dash_df["track_fit"] = (
    1.0 / (1.0 + dash_df["pp_avg_finish"].clip(lower=0))
    + 0.5 * dash_df["pp_win_rate"].fillna(0)
    if "pp_avg_finish" in dash_df.columns else np.nan
)

SIM_FEATURES = [
    "top_speed", "early_speed", "closing_speed", "track_fit",
    "gps_burst", "gps_stride_efficiency", "gps_advantage",
]

for col in SIM_FEATURES:
    dash_df[col] = pd.to_numeric(dash_df[col], errors="coerce")
    dash_df[f"{col}_norm"] = safe_z(dash_df[col].fillna(dash_df[col].median()))

dash_df["trad_model_score"] = safe_z(dash_df["trad_win_prob"])
dash_df["gps_model_score"]  = safe_z(dash_df["gps_win_prob"])
dash_df["model_score"]      = dash_df["gps_model_score"]

dash_df["model_early"]   = safe_z(dash_df["early_speed_norm"]          + 0.60 * dash_df["gps_burst_norm"])
dash_df["model_late"]    = safe_z(dash_df["closing_speed_norm"]         + 0.60 * dash_df["gps_stride_efficiency_norm"])
dash_df["model_stamina"] = safe_z(dash_df["track_fit_norm"]             + 0.35 * dash_df["gps_stride_efficiency_norm"])

dash_df["has_gps"]     = dash_df["gps_pp_count"].fillna(0) > 0
dash_df["explanation"] = dash_df.apply(make_horse_explanation, axis=1)

HORSE_COLS = [
    "race_label", "race_dt", "track_id", "race_number",
    "registration_number", "horse_name", "post_position", "morning_line_odds",
    "trad_win_prob", "gps_win_prob", "trad_top3_prob", "gps_top3_prob",
    "gps_advantage", "fair_odds_net",
    "has_gps", "gps_pp_count",
    "top_speed", "early_speed", "closing_speed", "track_fit",
    "gps_burst", "gps_stride_efficiency",
    "top_speed_norm", "early_speed_norm", "closing_speed_norm", "track_fit_norm",
    "gps_burst_norm", "gps_stride_efficiency_norm", "gps_advantage_norm",
    "trad_model_score", "gps_model_score", "model_score",
    "model_early", "model_late", "model_stamina", "explanation",
]
HORSE_COLS = [c for c in HORSE_COLS if c in dash_df.columns]

horse_data = (
    dash_df[HORSE_COLS]
    .sort_values(["race_dt", "track_id", "race_number", "gps_win_prob"],
                 ascending=[True, True, True, False])
    .reset_index(drop=True)
)

print(f"Simulator frame : {len(horse_data):,} entries | {horse_data['race_label'].nunique()} races")
print(f"GPS coverage    : {horse_data['has_gps'].mean():.1%} of horses have GPS PPs")
horse_data.head(6)

## 8 · Computation Helpers

In [ ]:

# ── SECTION 8: COMPUTATION HELPERS ──────────────────────────────────────

def build_race_input(selected_race_label, selected_horses=None):
    df = horse_data[horse_data["race_label"] == selected_race_label].copy()
    if selected_horses:
        df = df[df["horse_name"].isin(selected_horses)]
    return df.reset_index(drop=True)


def apply_user_weights(df, model_weight, gps_delta_weight,
                       w_top, w_early, w_closing, w_track,
                       w_gps_burst, w_gps_efficiency):
    temp = df.copy()
    temp["final_score"] = (
        model_weight       * temp["model_score"]
        + gps_delta_weight * temp["gps_advantage_norm"]
        + w_top            * temp["top_speed_norm"]
        + w_early          * temp["early_speed_norm"]
        + w_closing        * temp["closing_speed_norm"]
        + w_track          * temp["track_fit_norm"]
        + w_gps_burst      * temp["gps_burst_norm"]
        + w_gps_efficiency * temp["gps_stride_efficiency_norm"]
    )
    probs = softmax(temp["final_score"].fillna(0).values)
    temp["adjusted_win_prob"] = probs
    temp["adjusted_win_pct"]  = (100 * probs).round(1)
    return temp


def run_monte_carlo(df, n_sims=1000, noise_scale=0.35):
    """Run n_sims simulated races, track full finish-position distribution."""
    temp   = df.copy()
    horses = temp["horse_name"].tolist()
    win_counts  = {h: 0  for h in horses}
    top3_counts = {h: 0  for h in horses}
    finish_dist = {h: [] for h in horses}
    base = temp.set_index("horse_name")["final_score"].to_dict()
    for _ in range(n_sims):
        noisy  = {h: base[h] + np.random.normal(0, noise_scale) for h in horses}
        ranked = sorted(noisy.items(), key=lambda kv: kv[1], reverse=True)
        win_counts[ranked[0][0]] += 1
        for pos, (horse, _) in enumerate(ranked, start=1):
            finish_dist[horse].append(pos)
            if pos <= 3:
                top3_counts[horse] += 1
    temp["mc_win_pct"]    = temp["horse_name"].map(lambda h: round(100*win_counts[h]/n_sims, 1))
    temp["mc_top3_pct"]   = temp["horse_name"].map(lambda h: round(100*top3_counts[h]/n_sims, 1))
    temp["mc_avg_finish"] = temp["horse_name"].map(lambda h: round(np.mean(finish_dist[h]), 2))
    temp["finish_dist"]   = temp["horse_name"].map(lambda h: finish_dist[h])
    return temp


TRACK_COLORS = [
    "#FF6B6B","#4ECDC4","#FFD93D","#6BCB77",
    "#4D96FF","#FF922B","#CC5DE8","#F06595",
    "#74C0FC","#A9E34B","#FFA94D","#DA77F2",
]


def build_race_paths(df, n_frames=120):
    """Straight-track paths shaped by pace composites."""
    temp     = df.copy().reset_index(drop=True)
    n_horses = len(temp)
    lane_h   = 1.4
    horse_x  = np.zeros((n_horses, n_frames))
    horse_y  = np.zeros((n_horses, n_frames))
    for i in range(n_horses):
        y_pos    = i * lane_h
        strength = 0.25 * float(temp.loc[i, "adjusted_win_prob"])
        e_boost  = 0.15 * float(temp.loc[i, "model_early"]   or 0)
        s_boost  = 0.10 * float(temp.loc[i, "model_stamina"] or 0)
        l_boost  = 0.20 * float(temp.loc[i, "model_late"]    or 0)
        x = 0.0
        for t in range(n_frames):
            frac = t / n_frames
            if frac < 0.33:
                speed = 0.85 + e_boost + strength
            elif frac < 0.75:
                speed = 0.85 + s_boost + strength
            else:
                speed = 0.85 + l_boost + strength
            speed += np.random.normal(0, 0.08)
            x = min(x + max(speed, 0.3), 100.0)
            horse_x[i, t] = x
            horse_y[i, t] = y_pos
    return horse_x, horse_y


def build_race_figure(df):
    """Dark straight-track animation with horse emoji markers."""
    temp     = df.copy().reset_index(drop=True)
    n_horses = len(temp)
    n_frames = 120
    lane_h   = 1.4
    total_h  = (n_horses - 1) * lane_h
    horse_x, horse_y = build_race_paths(temp, n_frames=n_frames)

    fig = go.Figure()

    for i in range(n_horses):
        y   = i * lane_h
        col = TRACK_COLORS[i % len(TRACK_COLORS)]
        fig.add_shape(type="rect", x0=0, x1=100,
                      y0=y-0.55, y1=y+0.55,
                      fillcolor="rgba(255,255,255,0.025)",
                      line=dict(color="rgba(255,255,255,0.06)",width=1))
        fig.add_shape(type="rect", x0=-3, x1=-0.5,
                      y0=y-0.48, y1=y+0.48,
                      fillcolor=col, line=dict(color=col,width=0))
        fig.add_annotation(x=-4, y=y,
            text=f"<b>{temp.loc[i,'horse_name']}</b>",
            showarrow=False, xanchor="right",
            font=dict(size=11, color="rgba(255,255,255,0.8)", family="DM Sans"))

    # Finish line
    fig.add_shape(type="line", x0=100, x1=100, y0=-0.7, y1=total_h+0.7,
                  line=dict(color="#FFD700", width=3, dash="dot"))
    fig.add_annotation(x=100, y=total_h+0.95, text="FINISH",
                       showarrow=False,
                       font=dict(size=11, color="#FFD700", family="DM Sans"),
                       xanchor="center")

    # Start line
    fig.add_shape(type="line", x0=0, x1=0, y0=-0.7, y1=total_h+0.7,
                  line=dict(color="rgba(255,255,255,0.3)",width=2))

    # Initial markers
    for i in range(n_horses):
        fig.add_trace(go.Scatter(
            x=[horse_x[i,0]], y=[horse_y[i,0]],
            mode="text", text=["\U0001f434"],
            textfont=dict(size=20),
            name=temp.loc[i,"horse_name"],
            hovertemplate=(
                f"<b>{temp.loc[i,'horse_name']}</b><br>"
                f"GPS Win: {round(temp.loc[i,'gps_win_prob']*100,1)}%<br>"
                f"MC Win: {temp.loc[i,'mc_win_pct']}%"
                "<extra></extra>"
            ),
        ))

    # Animation frames
    frames = []
    for t in range(n_frames):
        fd = []
        for i in range(n_horses):
            fd.append(go.Scatter(
                x=[horse_x[i,t]], y=[horse_y[i,t]],
                mode="text", text=["\U0001f434"],
                textfont=dict(size=20),
                name=temp.loc[i,"horse_name"],
                hovertemplate=(
                    f"<b>{temp.loc[i,'horse_name']}</b><br>"
                    f"GPS Win: {round(temp.loc[i,'gps_win_prob']*100,1)}%<br>"
                    f"MC Win: {temp.loc[i,'mc_win_pct']}%"
                    "<extra></extra>"
                ),
            ))
        frames.append(go.Frame(data=fd, name=str(t)))

    fig.frames = frames
    fig.update_layout(
        height=max(300, 80 + n_horses*68),
        margin=dict(l=190, r=50, t=40, b=70),
        xaxis=dict(visible=False, range=[-5,104], showgrid=False, zeroline=False),
        yaxis=dict(visible=False, range=[-1,total_h+1.3], showgrid=False, zeroline=False),
        plot_bgcolor="#0d1117",
        paper_bgcolor="#0d1117",
        font=dict(color="white", family="DM Sans"),
        showlegend=False,
        updatemenus=[dict(
            type="buttons", showactive=False,
            bgcolor="#1c2333", bordercolor="#FFD700",
            font=dict(color="#FFD700"),
            x=0.5, xanchor="center", y=-0.14,
            buttons=[
                dict(label="  RACE!", method="animate",
                     args=[None, {"frame":{"duration":55,"redraw":True},
                                  "fromcurrent":True,"transition":{"duration":0}}]),
                dict(label="  Pause", method="animate",
                     args=[[None], {"mode":"immediate",
                                    "frame":{"duration":0,"redraw":False},
                                    "transition":{"duration":0}}]),
                dict(label="  Reset", method="animate",
                     args=[["0"], {"mode":"immediate",
                                   "frame":{"duration":0,"redraw":True},
                                   "transition":{"duration":0}}]),
            ],
        )],
        sliders=[dict(
            active=0,
            currentvalue={"prefix":"","font":{"color":"rgba(255,255,255,0.0)","size":1}},
            pad={"t":50},
            bgcolor="#1c2333",
            bordercolor="rgba(255,255,255,0.08)",
            tickcolor="rgba(255,255,255,0.2)",
            steps=[dict(
                args=[[str(t)],{"frame":{"duration":0,"redraw":True},
                                "mode":"immediate","transition":{"duration":0}}],
                label="", method="animate")
                for t in range(n_frames)],
        )],
    )
    return fig


def build_odds_figure(sim_df):
    """Horizontal bar chart: GPS vs traditional win prob."""
    df = sim_df.sort_values("mc_win_pct", ascending=True).reset_index(drop=True)
    n  = len(df)
    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=df["horse_name"], x=df["gps_win_prob"]*100,
        orientation="h", name="GPS Model",
        marker=dict(color=[TRACK_COLORS[i % len(TRACK_COLORS)] for i in range(n)],
                    opacity=0.9, line=dict(width=0)),
        text=[f"{v:.1f}%" for v in df["gps_win_prob"]*100],
        textposition="outside",
        textfont=dict(color="white", size=11, family="DM Sans"),
    ))
    fig.add_trace(go.Scatter(
        y=df["horse_name"], x=df["trad_win_prob"]*100,
        mode="markers", name="Traditional",
        marker=dict(symbol="diamond", size=10, color="white", opacity=0.6,
                    line=dict(width=1, color="rgba(255,255,255,0.25)")),
    ))
    fig.update_layout(
        title=dict(text="Win Probability   GPS vs Traditional",
                   font=dict(size=15, family="Playfair Display, serif", color="#FFD700"),
                   x=0.5, xanchor="center"),
        barmode="overlay",
        height=max(260, 60+n*42),
        margin=dict(l=20, r=70, t=55, b=40),
        plot_bgcolor="#0d1117", paper_bgcolor="#161b27",
        font=dict(color="rgba(255,255,255,0.8)", family="DM Sans"),
        xaxis=dict(title="Win Probability (%)", gridcolor="rgba(255,255,255,0.05)",
                   zeroline=False, showline=False,
                   tickfont=dict(color="rgba(255,255,255,0.4)")),
        yaxis=dict(showgrid=False, zeroline=False,
                   tickfont=dict(color="rgba(255,255,255,0.8)", size=12)),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.1,
                    font=dict(size=11, color="rgba(255,255,255,0.6)"),
                    bgcolor="rgba(0,0,0,0)"),
    )
    return fig


def build_distribution_figure(sim_df):
    """
    Finish-position heatmap.
    Each cell = % of 1,000 sims that horse finished in that position.
    Much richer than a single win% number.
    """
    df     = sim_df.sort_values("mc_win_pct", ascending=False).reset_index(drop=True)
    horses = df["horse_name"].tolist()
    n      = len(horses)
    n_sims = len(df["finish_dist"].iloc[0])

    matrix = np.zeros((n, n))
    for i, horse in enumerate(horses):
        dist = df.loc[df["horse_name"]==horse, "finish_dist"].iloc[0]
        for pos in dist:
            matrix[i, int(pos)-1] += 1
    matrix = 100 * matrix / n_sims

    pos_labels = []
    for p in range(n):
        if   p == 0: pos_labels.append("1st")
        elif p == 1: pos_labels.append("2nd")
        elif p == 2: pos_labels.append("3rd")
        else:        pos_labels.append(f"{p+1}th")

    fig = go.Figure(data=go.Heatmap(
        z=matrix,
        x=pos_labels,
        y=horses,
        colorscale=[
            [0.0,  "#0d1117"],
            [0.08, "#1a3a5c"],
            [0.25, "#1a6b4a"],
            [0.55, "#d4a017"],
            [1.0,  "#FFD700"],
        ],
        showscale=True,
        colorbar=dict(title="%",
                      titlefont=dict(color="rgba(255,255,255,0.5)"),
                      tickfont=dict(color="rgba(255,255,255,0.5)"),
                      bgcolor="rgba(0,0,0,0)",
                      bordercolor="rgba(255,255,255,0.08)"),
        text=[[f"{v:.0f}%" if v > 2 else "" for v in row] for row in matrix],
        texttemplate="%{text}",
        textfont=dict(size=11, color="white"),
        hoverongaps=False,
        hovertemplate="<b>%{y}</b><br>Finished %{x}: <b>%{z:.1f}%</b> of sims<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text="Finish-Position Distribution   (1,000 simulations)",
                   font=dict(size=15, family="Playfair Display, serif", color="#FFD700"),
                   x=0.5, xanchor="center"),
        height=max(280, 100+n*48),
        margin=dict(l=20, r=80, t=60, b=40),
        plot_bgcolor="#0d1117", paper_bgcolor="#161b27",
        font=dict(color="rgba(255,255,255,0.8)", family="DM Sans"),
        xaxis=dict(side="top", tickfont=dict(size=13), showgrid=False),
        yaxis=dict(showgrid=False, tickfont=dict(size=12), autorange="reversed"),
    )
    return fig


print("Computation helpers ready.")


## 9 · Reimagined Dashboard

In [ ]:

import dash
from dash import dcc, html, Input, Output, State

# ── SECTION 9: DASH APP ──────────────────────────────────────────────────────

_CSS = Path("css_block.txt").read_text(encoding="utf-8")
_JS  = Path("js_block.txt").read_text(encoding="utf-8")

available_races = horse_data["race_label"].drop_duplicates().sort_values().tolist()
default_race    = available_races[0]
default_horses  = horse_data.loc[horse_data["race_label"] == default_race, "horse_name"].tolist()


def make_weight_row(slider_id, label, default=1.0):
    return html.Div([
        html.Span(label, className="weight-label"),
        html.Div(dcc.Slider(id=slider_id, min=0, max=3, step=0.1, value=default,
                            marks=None, tooltip={"placement":"bottom","always_visible":False}),
                 style={"flex":"1"}),
        html.Span(str(default), id=f"{slider_id}_val", className="weight-value"),
    ], className="weight-row")


def horse_card(row, rank, color):
    gps_adv   = float(row.get("gps_advantage", 0) or 0)
    gps_adv_p = round(gps_adv * 100, 1)
    has_gps   = bool(row.get("has_gps", False))

    if not has_gps:
        badge_cls, badge_txt = "no-data", "No GPS data"
    elif gps_adv_p > 1.5:
        badge_cls, badge_txt = "bullish", f"GPS +{gps_adv_p}%"
    elif gps_adv_p < -1.5:
        badge_cls, badge_txt = "bearish", f"GPS {gps_adv_p}%"
    else:
        badge_cls, badge_txt = "neutral", "GPS neutral"

    medal   = {1:"🥇",2:"🥈",3:"🥉"}.get(rank, f"#{rank}")
    win_pct = round(float(row["mc_win_pct"]), 1)
    top3_p  = round(float(row["mc_top3_pct"]), 1)
    gps_wp  = round(float(row["gps_win_prob"])*100, 1)
    trad_wp = round(float(row["trad_win_prob"])*100, 1)
    avg_fin = float(row["mc_avg_finish"])
    ml_odds = row.get("morning_line_odds", "—")
    post    = row.get("post_position", "—")
    expl    = row.get("explanation", "")

    return html.Div([
        html.Div(style={"position":"absolute","top":0,"left":0,"width":"4px",
                        "height":"100%","background":color,
                        "borderRadius":"14px 0 0 14px"}),
        html.Div(str(rank), className="horse-card-rank"),

        html.Div([
            html.Span(f"{medal} ", style={"fontSize":"15px"}),
            html.Span(row["horse_name"], className="horse-card-name",
                      style={"display":"inline"}),
        ]),
        html.Div(f"Post {post}  ·  ML {ml_odds}  ·  {'GPS' if has_gps else 'No GPS'}",
                 className="horse-card-meta"),
        html.Span(badge_txt, className=f"gps-badge {badge_cls}"),

        html.Div([
            html.Div([html.Span("Win %", className="horse-stat-label"),
                      html.Span(f"{win_pct}%", className="horse-stat-value",
                                style={"color":color})],
                     className="horse-stat-row"),
            html.Div(className="prob-bar-track", children=[
                html.Div(className="prob-bar-fill",
                         style={"width":f"{min(win_pct*2,100)}%","background":color}),
            ]),
        ], style={"marginBottom":"8px","marginTop":"4px"}),

        html.Div([
            html.Div([html.Span("Top-3 %",    className="horse-stat-label"),
                      html.Span(f"{top3_p}%", className="horse-stat-value")], className="horse-stat-row"),
            html.Div([html.Span("GPS Win %",   className="horse-stat-label"),
                      html.Span(f"{gps_wp}%",  className="horse-stat-value")], className="horse-stat-row"),
            html.Div([html.Span("Trad Win %",  className="horse-stat-label"),
                      html.Span(f"{trad_wp}%", className="horse-stat-value",
                                style={"color":"rgba(255,255,255,0.35)"})], className="horse-stat-row"),
            html.Div([html.Span("Avg Finish",  className="horse-stat-label"),
                      html.Span(str(avg_fin),  className="horse-stat-value")], className="horse-stat-row"),
        ]),

        html.Div(expl, className="explanation-text") if expl else None,
    ], className="horse-card", style={"position":"relative"})


INDEX_STRING = """<!DOCTYPE html>
<html>
<head>
    {%metas%}
    <title>{%title%}</title>
    {%favicon%}
    {%css%}
    <style>""" + "{CSS_PLACEHOLDER}" + """</style>
</head>
<body>
    {%app_entry%}
    <footer>{%config%}{%scripts%}{%renderer%}</footer>
    <script>""" + "{JS_PLACEHOLDER}" + """</script>
</body>
</html>"""

INDEX_STRING = INDEX_STRING.replace("{CSS_PLACEHOLDER}", _CSS).replace("{JS_PLACEHOLDER}", _JS)


app = dash.Dash(__name__, title="PostTime  GPS Race Intelligence",
                suppress_callback_exceptions=True)
app.index_string = INDEX_STRING


app.layout = html.Div([

    # Loading overlay
    html.Div([
        html.Div("🐴", className="loading-horse"),
        html.Div("Running the Race", className="loading-title"),
        html.Div("SIMULATING  ·  SCORING  ·  RENDERING", className="loading-sub"),
        html.Div(className="loading-bar-track",
                 children=[html.Div(id="loading-fill", className="loading-bar-fill")]),
        html.Div("Preparing...", id="loading-step", className="loading-step"),
    ], id="loading-overlay", className="loading-overlay"),

    # Header
    html.Div([
        html.Div("🏇  POSTTIME", className="app-title"),
        html.Div("GPS-Enhanced Race Intelligence  ·  Upcoming Race Handicapper",
                 className="app-subtitle"),
    ], className="app-header"),

    # Body
    html.Div([

        # Sidebar
        html.Div([
            html.Div([
                html.Span("Select Race", className="sidebar-label"),
                dcc.Dropdown(id="race_selector",
                             options=[{"label":r,"value":r} for r in available_races],
                             value=default_race, clearable=False,
                             style={"marginBottom":"14px"}),
                html.Span("Horses", className="sidebar-label"),
                dcc.Dropdown(id="horse_selector",
                             options=[{"label":h,"value":h} for h in default_horses],
                             value=default_horses, multi=True),
            ], className="sidebar-section"),

            html.Hr(className="section-divider"),

            html.Div([
                html.Span("Model Weights", className="sidebar-label"),
                make_weight_row("model_weight_slider", "GPS Model",     2.0),
                make_weight_row("gps_delta_slider",    "GPS Advantage", 1.0),
            ], className="sidebar-section"),

            html.Div([
                html.Span("Pace Weights", className="sidebar-label"),
                make_weight_row("top_slider",       "Top Speed",      1.0),
                make_weight_row("early_slider",     "Early Speed",    1.0),
                make_weight_row("closing_slider",   "Closing Speed",  1.0),
                make_weight_row("track_slider",     "Track Fit",      1.0),
                make_weight_row("gps_burst_slider", "GPS Burst",      1.0),
                make_weight_row("gps_eff_slider",   "GPS Efficiency", 1.0),
            ], className="sidebar-section"),

            html.Hr(className="section-divider"),

            html.Button("RUN SIMULATION", id="run_button", n_clicks=0,
                        className="run-btn"),
            html.Div(id="sim-meta",
                     style={"marginTop":"10px","fontSize":"11px",
                            "color":"rgba(255,255,255,0.22)","textAlign":"center"}),
        ], className="sidebar"),

        # Main output
        html.Div([
            html.Div([
                html.Button("🏇  Race",          className="tab-btn active",
                            **{"data-tab":"race"},
                            id="tab-btn-race",    n_clicks=0),
                html.Button("📊  Probabilities",  className="tab-btn",
                            **{"data-tab":"probs"},
                            id="tab-btn-probs",   n_clicks=0),
                html.Button("🎲  Distributions",  className="tab-btn",
                            **{"data-tab":"dist"},
                            id="tab-btn-dist",    n_clicks=0),
                html.Button("🐴  Horse Cards",    className="tab-btn",
                            **{"data-tab":"cards"},
                            id="tab-btn-cards",   n_clicks=0),
            ], className="tab-bar"),

            html.Div(dcc.Graph(id="race_graph",
                               config={"displayModeBar":False},
                               style={"borderRadius":"10px","overflow":"hidden"}),
                     id="tab-race",  className="tab-panel"),

            html.Div(dcc.Graph(id="comparison_chart",
                               config={"displayModeBar":False},
                               style={"borderRadius":"10px","overflow":"hidden"}),
                     id="tab-probs", className="tab-panel",
                     style={"display":"none"}),

            html.Div(dcc.Graph(id="dist_chart",
                               config={"displayModeBar":False},
                               style={"borderRadius":"10px","overflow":"hidden"}),
                     id="tab-dist",  className="tab-panel",
                     style={"display":"none"}),

            html.Div(html.Div(id="horse_cards", className="horse-cards"),
                     id="tab-cards", className="tab-panel",
                     style={"display":"none"}),

        ], className="main-content"),

    ], className="app-body"),

], style={"minHeight":"100vh"})


# Callbacks

@app.callback(
    [Output("horse_selector","options"), Output("horse_selector","value")],
    Input("race_selector","value"),
)
def update_horse_options(selected_race):
    horses = horse_data.loc[horse_data["race_label"]==selected_race,"horse_name"].tolist()
    return [{"label":h,"value":h} for h in horses], horses


for _sid in ["model_weight_slider","gps_delta_slider","top_slider",
             "early_slider","closing_slider","track_slider",
             "gps_burst_slider","gps_eff_slider"]:
    @app.callback(Output(f"{_sid}_val","children"), Input(_sid,"value"))
    def _upd(v): return str(round(v,1))


@app.callback(
    [Output("race_graph","figure"), Output("comparison_chart","figure"),
     Output("dist_chart","figure"), Output("horse_cards","children"),
     Output("sim-meta","children")],
    Input("run_button","n_clicks"),
    [State("race_selector","value"),       State("horse_selector","value"),
     State("model_weight_slider","value"), State("gps_delta_slider","value"),
     State("top_slider","value"),          State("early_slider","value"),
     State("closing_slider","value"),      State("track_slider","value"),
     State("gps_burst_slider","value"),    State("gps_eff_slider","value")],
)
def run_simulation(n_clicks, selected_race, selected_horses,
                   model_weight, gps_delta_weight,
                   w_top, w_early, w_closing, w_track, w_gps_burst, w_gps_eff):
    def blank(msg="Select 2 or more horses, then click RUN SIMULATION"):
        return go.Figure().update_layout(
            annotations=[dict(text=msg, showarrow=False,
                              font=dict(size=14,color="rgba(255,255,255,0.25)",
                                        family="DM Sans"),
                              xref="paper",yref="paper",x=0.5,y=0.5)],
            plot_bgcolor="#0d1117", paper_bgcolor="#0d1117",
        )

    if not selected_horses or len(selected_horses) < 2:
        return blank(), blank(), blank(), [], ""

    race_df = build_race_input(selected_race, selected_horses)
    scored  = apply_user_weights(race_df, model_weight, gps_delta_weight,
                                  w_top, w_early, w_closing, w_track,
                                  w_gps_burst, w_gps_eff)
    sim_df  = run_monte_carlo(scored, n_sims=1000)
    sim_df  = sim_df.sort_values("mc_win_pct", ascending=False).reset_index(drop=True)

    race_fig = build_race_figure(sim_df)
    odds_fig = build_odds_figure(sim_df)
    dist_fig = build_distribution_figure(sim_df)

    cards = [
        horse_card(row, rank+1, TRACK_COLORS[rank % len(TRACK_COLORS)])
        for rank, (_, row) in enumerate(sim_df.iterrows())
    ]

    n_gps = int(sim_df["has_gps"].sum()) if "has_gps" in sim_df.columns else 0
    meta  = f"1,000 simulations  ·  {n_gps} of {len(sim_df)} horses have GPS data"
    return race_fig, odds_fig, dist_fig, cards, meta


print("App ready.  Launch: app.run(debug=False, port=8052)")


## 10 · Launch

In [ ]:
app.run(debug=False, port=8052)